# IndoBERTweet + weighted loss — BALANCED 3000 (Track C)

Coba angkat akurasi di atas baseline IndoBERT (0,7733) pada dataset **domain-aware**.
Dua perbaikan:
1. **Model `indolem/indobertweet-base-uncased`** — dilatih pada **Twitter Indonesia**
   (bahasa medsos/alay), domain lebih cocok utk komentar YouTube daripada indobert-base
   (Wikipedia/berita). *Kandidat lompatan terbesar.*
2. **Weighted cross-entropy** (bobot kelas terbalik thd frekuensi).

**Self-contained** (cukup `MONGO_URI`): baca `processed_bert` via flag `in_balanced3k=True`
(label domain-aware terbaru). Split kanonik identik SVM/IndoBERT (seed=42).
Runtime -> **T4 GPU** -> Run all. Output `indobertweet_balanced3k_metrics.json` ter-download.

Pembanding adil: set `MODEL_NAME='indobenchmark/indobert-base-p1'` utk mengukur efek
weighted-loss SAJA (tanpa ganti model).

In [ ]:
%pip install -q "transformers>=4.40" torch "pymongo[srv]" dnspython certifi scikit-learn matplotlib pandas numpy
import torch; print("CUDA:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (lambat!)")

In [ ]:
import os, pandas as pd
from pymongo import MongoClient
import certifi
from sklearn.model_selection import train_test_split

FLAG = "in_balanced3k"
MODEL_NAME = "indolem/indobertweet-base-uncased"   # pembanding: "indobenchmark/indobert-base-p1"
TAG = "balanced3k"

MONGO_URI = os.environ.get("MONGO_URI", "")
if not MONGO_URI:
    try:
        from google.colab import userdata; MONGO_URI = userdata.get("MONGO_URI")
    except Exception: MONGO_URI = ""
if not MONGO_URI:
    from getpass import getpass; MONGO_URI = getpass("MONGO_URI: ")
DB = os.environ.get("MONGO_DB_NAME", "youtube_sentiment")
client = MongoClient(MONGO_URI, tlsCAFile=certifi.where(), serverSelectionTimeoutMS=20000)
client.admin.command("ping"); print("Mongo OK.")

LABELS = ["Negatif", "Netral", "Positif"]; LABEL2ID = {l: i for i, l in enumerate(LABELS)}
df = pd.DataFrame(list(client[DB]["processed_bert"].find({FLAG: True}, {"_id":0,"comment_id":1,"bert":1,"label":1})))
assert len(df), f"Tak ada dokumen ber-flag {FLAG}=True."
df["label_id"] = df["label"].map(LABEL2ID)
df = df.sort_values("comment_id").reset_index(drop=True)
tmp, df_test = train_test_split(df, test_size=0.10, stratify=df["label_id"], random_state=42)
df_train, df_val = train_test_split(tmp, test_size=0.20/0.90, stratify=tmp["label_id"], random_state=42)
print(f"{MODEL_NAME} | n={len(df)} train={len(df_train)} val={len(df_val)} test={len(df_test)}")

In [ ]:
from transformers import AutoTokenizer
import torch
MAX_LEN, SEED = 128, 42
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
class DS(torch.utils.data.Dataset):
    def __init__(self, texts, labels):
        self.enc = tok(list(texts), truncation=True, max_length=MAX_LEN, padding=True); self.labels = list(labels)
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        it = {k: torch.tensor(v[i]) for k, v in self.enc.items()}; it["labels"] = torch.tensor(self.labels[i]); return it
ds_train = DS(df_train["bert"].astype(str), df_train["label_id"])
ds_val   = DS(df_val["bert"].astype(str),   df_val["label_id"])
ds_test  = DS(df_test["bert"].astype(str),  df_test["label_id"])
print("Dataset siap.")

In [ ]:
import numpy as np
from collections import Counter
from transformers import (AutoModelForSequenceClassification, TrainingArguments, Trainer,
                          EarlyStoppingCallback, set_seed)
from sklearn.metrics import f1_score, accuracy_score
set_seed(SEED)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3, id2label={i:l for i,l in enumerate(LABELS)}, label2id=LABEL2ID)

cnt = Counter(df_train["label_id"]); n = len(df_train)
class_w = torch.tensor([n/(3*cnt[i]) for i in range(3)], dtype=torch.float)
print("class weights:", [round(x,3) for x in class_w.tolist()])

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop("labels"); out = model(**inputs)
        loss = torch.nn.functional.cross_entropy(out.logits, labels, weight=class_w.to(out.logits.device))
        return (loss, out) if return_outputs else loss

def compute_metrics(p):
    pr = np.argmax(p.predictions, axis=1)
    return {"macro_f1": f1_score(p.label_ids, pr, average="macro"), "accuracy": accuracy_score(p.label_ids, pr)}

_kw = dict(output_dir="out", num_train_epochs=6, per_device_train_batch_size=16, per_device_eval_batch_size=32,
           learning_rate=2e-5, weight_decay=0.01, warmup_ratio=0.1, save_total_limit=2,
           load_best_model_at_end=True, metric_for_best_model="macro_f1", greater_is_better=True,
           seed=SEED, logging_steps=50, report_to="none")
try:
    args = TrainingArguments(eval_strategy="epoch", save_strategy="epoch", **_kw)
except TypeError:
    args = TrainingArguments(evaluation_strategy="epoch", save_strategy="epoch", **_kw)
trainer = WeightedTrainer(model=model, args=args, train_dataset=ds_train, eval_dataset=ds_val,
                          compute_metrics=compute_metrics, callbacks=[EarlyStoppingCallback(early_stopping_patience=3)])
print("Trainer siap.")

In [ ]:
trainer.train(); print("Val terbaik:", trainer.evaluate())

In [ ]:
import json
from sklearn.metrics import classification_report, confusion_matrix
pred = trainer.predict(ds_test)
y_pred = np.argmax(pred.predictions, axis=1).tolist(); y_true = df_test["label_id"].tolist()
def evaluate(yt, yp, labels=LABELS):
    ids = list(range(len(labels)))
    rep = classification_report(yt, yp, labels=ids, target_names=labels, output_dict=True, zero_division=0)
    return {"accuracy": round(accuracy_score(yt,yp),4), "macro_f1": round(f1_score(yt,yp,average="macro",zero_division=0),4),
            "weighted_f1": round(f1_score(yt,yp,average="weighted",zero_division=0),4),
            "per_class": {l:{k:round(rep[l][k],4) for k in ["precision","recall","f1-score"]} | {"support":int(rep[l]["support"])} for l in labels},
            "confusion_matrix": confusion_matrix(yt,yp,labels=ids).tolist(), "labels": labels}
m = evaluate(y_true, y_pred)
print("="*56); print(f"  {MODEL_NAME}  [{TAG}] TEST"); print("="*56)
print(f"  Accuracy : {m['accuracy']:.4f}  | macro-F1: {m['macro_f1']:.4f}  (baseline IndoBERT acc=0.7733)")
for l in LABELS:
    pc = m["per_class"][l]; print(f"    {l:<8} P={pc['precision']:.3f} R={pc['recall']:.3f} F1={pc['f1-score']:.3f}")

json.dump({"model": MODEL_NAME, "dataset": TAG, "test": m},
          open("indobertweet_balanced3k_metrics.json", "w"), ensure_ascii=False, indent=2)
# predictions
logits = np.asarray(pred.predictions, dtype=float); e = np.exp(logits - logits.max(axis=1, keepdims=True))
pdf = pd.DataFrame({"comment_id": df_test["comment_id"].to_numpy(), "text": df_test["bert"].to_numpy(),
                    "label_asli": df_test["label"].to_numpy(), "prediksi": [LABELS[i] for i in y_pred]})
pdf["benar"] = pdf["label_asli"] == pdf["prediksi"]; pdf["keyakinan"] = np.round((e/e.sum(axis=1,keepdims=True)).max(axis=1),4)
pdf.to_csv("indobertweet_balanced3k_predictions.csv", index=False)
import matplotlib.pyplot as plt
cm = np.array(m["confusion_matrix"]); fig, ax = plt.subplots(figsize=(5,4.3)); ax.imshow(cm, cmap="Purples")
ax.set_xticks(range(3), LABELS); ax.set_yticks(range(3), LABELS); ax.set_xlabel("Prediksi"); ax.set_ylabel("Aktual")
ax.set_title(f"IndoBERTweet — Test (acc={m['accuracy']:.3f})")
th = cm.max()/2
for i in range(3):
    for j in range(3): ax.text(j,i,cm[i,j],ha='center',va='center',color='white' if cm[i,j]>th else 'black')
fig.tight_layout(); fig.savefig("indobertweet_balanced3k_test_confusion.png", dpi=120); plt.show()
print("Tersimpan: indobertweet_balanced3k_metrics.json (+ predictions, confusion)")
try:
    from google.colab import files
    files.download("indobertweet_balanced3k_metrics.json")
    files.download("indobertweet_balanced3k_test_confusion.png")
    files.download("indobertweet_balanced3k_predictions.csv")
except Exception as ex:
    print("Download manual dari panel Files:", ex)